# Reformat README and Generate Screenshots
This notebook rewrites the repository README to a docker-mailserver-style layout and generates admin UI screenshot placeholders.

In [ ]:
import os
from pathlib import Path
from typing import List

try:
    from PIL import Image, ImageDraw, ImageFont
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pillow'])
    from PIL import Image, ImageDraw, ImageFont

In [ ]:
root = Path('.')
readme_path = root / 'README.md'
current_readme = readme_path.read_text(encoding='utf-8') if readme_path.exists() else ''
print('Loaded README length:', len(current_readme))

In [ ]:
screenshot_dir = root / 'docs' / 'screenshots'
screenshot_dir.mkdir(parents=True, exist_ok=True)

try:
    font = ImageFont.truetype('arial.ttf', 18)
except Exception:
    font = ImageFont.load_default()


def make_screenshot(path: Path, title: str, rows: List[str]):
    width, height = 1080, 640
    img = Image.new('RGB', (width, height), '#f8fafc')
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 0, width, 100], fill='#111827')
    draw.text((40, 34), title, fill='#ffffff', font=font)
    y = 140
    for row in rows:
        draw.rectangle([40, y, width - 40, y + 96], fill='#ffffff', outline='#cbd5e1', width=2)
        draw.text((60, y + 28), row, fill='#111827', font=font)
        y += 120
    img.save(path)

make_screenshot(
    screenshot_dir / 'admin-dashboard.png',
    'Admin Dashboard',
    [
        'Protected admin login',
        'Manage DNS settings and API keys',
        'View current configuration at a glance',
    ],
)
make_screenshot(
    screenshot_dir / 'api-keys.png',
    'API Key Management',
    [
        'Generate and revoke API keys',
        'View key status and labels',
        'Protect the REST endpoint with bearer tokens',
    ],
)
make_screenshot(
    screenshot_dir / 'settings.png',
    'DNS Settings',
    [
        'Target DNS server and zone',
        'Microsoft DNS username/password stored encrypted',
        'Update values in the admin UI',
    ],
)
print('screenshots created')

In [ ]:
new_readme = '''# Microsoft DNS REST Service

[![Docker](https://img.shields.io/badge/docker-ready-blue)](https://www.docker.com/) [![Python](https://img.shields.io/badge/python-3.12-green)](https://www.python.org/)

A Dockerized FastAPI service to manage Microsoft DNS records through a protected web admin interface and secure API key authentication.

## Features

- Protected admin UI for login and settings management
- Generate and revoke API keys for the REST API
- Store target DNS server, zone, and Microsoft DNS credentials encrypted in the database
- Create or update DNS records with existence checking
- Docker Compose-ready deployment

## Screenshots

![Admin Dashboard](docs/screenshots/admin-dashboard.png)

![API Key Management](docs/screenshots/api-keys.png)

![DNS Settings](docs/screenshots/settings.png)

## Quick Start

1. Copy `.env.example` to `.env`.
2. Edit `.env` and set `SECRET_KEY`, `ENCRYPTION_KEY`, `ADMIN_USER`, and `ADMIN_PASSWORD`.
3. Run:

```bash
docker compose up --build
```

4. Open the admin UI at `http://localhost:8000/login`.

## Configuration

The following variables are supported in `.env`:

- `SECRET_KEY` — secret used to sign session cookies
- `ENCRYPTION_KEY` — key used to encrypt settings at rest
- `ADMIN_USER` — initial admin username
- `ADMIN_PASSWORD` — initial admin password
- `DATABASE_URL` — optional path to the database (default `sqlite:///./data/app.db`)
- `AZURE_TENANT_ID`, `AZURE_CLIENT_ID`, `AZURE_CLIENT_SECRET`, `AZURE_SUBSCRIPTION_ID` — optional Azure DNS credentials

## Admin UI

Access the web admin interface to:

- manage DNS settings and target zone
- store Microsoft DNS username/password securely
- create and revoke API keys

## API Usage

Use `X-API-Key` or `Authorization: Bearer <key>` when calling the record endpoint:

```bash
curl -X POST http://localhost:8000/dns-record \
  -H "Content-Type: application/json" \
  -H "X-API-Key: <your-api-key>" \
  -d '{
    "subscription_id": "<subscription-id>",
    "resource_group": "<resource-group>",
    "zone_name": "example.com",
    "record_type": "A",
    "record_name": "www",
    "ttl": 300,
    "values": ["192.0.2.1"]
  }'
```

## Repository Layout

- `Dockerfile` — container build settings
- `docker-compose.yml` — local deployment definition
- `src/` — application source code, templates, and static assets
- `docs/screenshots/` — admin UI screenshot placeholders
- `requirements.txt` — Python dependencies

## License

MIT
'''

readme_path.write_text(new_readme, encoding='utf-8')
print('README regenerated')